# 📘 부록: 행렬 연산과 텐서 모양 바꾸기

트랜스포머 코드의 줄 대부분은 텐서의 모양(shape)을 바꾸거나, 모양을 맞춰 계산하는 줄입니다. 이 부록은 숫자 하나짜리 텐서에서 시작해 교안 01의 `(B, W_L, D)` 텐서까지, 연산 하나마다 그림과 작은 예시로 값이 어디로 가는지 따라갑니다. 교안 01의 모양 주석 `# (배치, 토큰 수, d_model)`이 막힐 때 해당 절로 돌아와 봅니다.

**이 부록을 마치면 할 수 있는 것**
1. 텐서의 shape, 축 번호, 인덱싱 결과의 모양을 **읽습니다**.
2. 행렬 곱을 칸마다 내적 하나로 **계산하고**, 토큰끼리의 내적 점수에 softmax를 씌워 비율로 **바꿉니다**.
3. softmax로 만든 비율로 토큰 벡터를 섞는 가중합을 **계산합니다**.
4. transpose, reshape, squeeze, unsqueeze가 값을 어디로 옮기는지 **설명합니다**.

**표기:** `(B, W_L, D)` = 교안 01 모양 주석의 `(배치, 토큰 수, d_model)`
- **B:** 배치(batch) 크기. 한 번에 넣는 문장 수입니다.
- **W_L:** 문장 하나의 토큰 수(시퀀스 길이)입니다. 교안 01 코드의 `n_tokens = tokens.shape[1]`이고, 패딩으로 길이를 맞춘 뒤에는 `max_len`(64)과 같습니다. 가중치 `W`와는 관계없는 이름입니다.
- **D:** 토큰 벡터 하나의 길이(숫자 개수). 교안 01의 `d_model`입니다.
- **성분(component):** 벡터 안의 숫자 하나입니다. 교안 01이 "64차원 벡터"라고 부르는 토큰 벡터는 성분이 64개입니다.
- **텐서의 차원과 벡터의 차원:** "D차원 벡터"는 성분이 D개인 1차원 텐서입니다. 텐서의 차원(ndim, 축 수)과 벡터의 차원(성분 수)은 뜻이 달라, 이 부록은 뒤의 것을 "길이"라고 부릅니다.
- 이 부록의 예시는 `(B, W_L, D) = (2, 5, 4)`이고 교안 01은 `(128, 64, 64)`입니다. 세 크기를 모두 다르게 골라, 축을 잘못 고르면 출력 모양이 달라져 바로 보이게 했습니다.

## 1. 실습 준비

데이터 파일은 쓰지 않습니다. 텐서를 손으로 만들어 모양과 값을 함께 출력합니다.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn

# 텐서를 출력할 때 소수 둘째 자리까지, 지수 표기 없이 보이게 합니다
torch.set_printoptions(precision=2, sci_mode=False)
print("torch 버전:", torch.__version__)

## 2. 텐서의 모양

### 2-1. 0차원부터 3차원까지

**텐서(tensor)**
- 같은 자료형의 숫자를 축 0개 이상으로 늘어놓은 배열입니다(축이 0개면 숫자 하나). 숫자를 늘어놓은 방향 하나를 **축(axis)** 이라 하고, PyTorch 함수에서는 `dim`이라는 인자 이름으로 축을 고릅니다.
- **shape:** 축마다 몇 칸인지 적은 튜플입니다. `(2, 3, 4)`는 축 0이 2칸, 축 1이 3칸, 축 2가 4칸입니다.
- **ndim:** 축의 개수입니다. shape 튜플의 길이와 같습니다.
- **원소 수(`numel()`):** shape의 크기를 모두 곱한 값입니다. `(2, 3, 4)`이면 24개입니다.

<img src="images/71_tensor_dims.png" width="1050" alt="제목은 '0차원부터 3차원까지: 축이 하나씩 늘어납니다'입니다. 네 판이 왼쪽부터 놓여 있습니다. '0차원 스칼라 S'는 3 한 칸과 '축 없음', 'shape ()', 'ndim 0'이 적혀 있습니다. '1차원 벡터 V'는 1, 2, 3, 4 네 칸과 가로 화살표 '축 0 (4개)', 'shape (4,)', 'ndim 1'입니다. '2차원 행렬 M'은 0~11이 적힌 3행 4열 표에 세로 화살표 '축 0 (행 3개)', 가로 화살표 '축 1 (열 4개)', 'shape (3, 4)', 'ndim 2'입니다. '3차원 텐서 T'는 큐브로 그려져 있고 앞면에 0~11, 윗면에 12~15, 옆면에 15, 19, 23이 보이며 '뒤 장 T[1] 12~23', '앞 장 T[0] 0~11'이 표시되어 있습니다. 화살표는 '축 0: 2장 (깊이)', '축 1 행 3개', '축 2: 열 4개'이고 'shape (2, 3, 4)', 'ndim 3'이 적혀 있습니다. 아래 띠에 'shape = 축마다의 크기, ndim = 축의 개수', '원소 수 = shape의 곱 (2 * 3 * 4 = 24)'가 적혀 있습니다.">

그림의 네 텐서를 그대로 만듭니다. 대괄호가 한 겹 늘 때마다 축이 하나 늘어납니다.

In [ ]:
# 0차원(스칼라), 1차원(벡터), 2차원(행렬), 3차원 텐서
S = torch.tensor(3.0)
V = torch.tensor([1.0, 2.0, 3.0, 4.0])

# 행렬: 숫자 4개짜리 행 3개 -> (3, 4)
M = torch.tensor([[0, 1, 2, 3],
                  [4, 5, 6, 7],
                  [8, 9, 10, 11]])

# 3차원: (3, 4) 판 2장 -> (2, 3, 4)
T = torch.tensor([[[0, 1, 2, 3],
                   [4, 5, 6, 7],
                   [8, 9, 10, 11]],
                  [[12, 13, 14, 15],
                   [16, 17, 18, 19],
                   [20, 21, 22, 23]]])

# 텐서마다 shape, ndim, 원소 수를 표 한 장으로 비교합니다
rows = []
for name, tensor in [("S", S), ("V", V), ("M", M), ("T", T)]:
    rows.append({"이름": name, "shape": tuple(tensor.shape), "ndim": tensor.ndim, "원소 수": tensor.numel()})
display(pd.DataFrame(rows))
print("T =")
print(T)

**결과 읽기**
- **S는 shape `()`, ndim 0:** 축이 없는 숫자 하나입니다. 손실값처럼 숫자 하나를 담는 텐서가 이 모양입니다.
- **T는 shape `(2, 3, 4)`, 원소 24개:** 출력에서 바깥 대괄호 안에 `(3, 4)` 판이 2장 있습니다. 출력의 대괄호 깊이가 ndim과 같습니다.

### 2-2. 축 번호와 인덱싱

2-1에서 만든 3차원 텐서 `T`(tensor의 첫 글자)로 축 번호를 봅니다. `T`는 숫자 0~23을 3행 4열 판 2장에 나눠 담은 예시입니다.

```text
T = [[[ 0,  1,  2,  3],     <- T[0]: 첫 번째 판 (3, 4)
      [ 4,  5,  6,  7],
      [ 8,  9, 10, 11]],

     [[12, 13, 14, 15],     <- T[1]: 두 번째 판 (3, 4)
      [16, 17, 18, 19],
      [20, 21, 22, 23]]]
```

- **축 0 (크기 2):** 판 번호입니다. `T[0]`은 첫 번째 판(0~11), `T[1]`은 두 번째 판(12~23)입니다. 그림에서는 큐브의 앞 장과 뒤 장입니다.
- **축 1 (크기 3):** 판 안의 행 번호, **축 2 (크기 4):** 행 안의 열 번호입니다.
- **값 하나 고르기:** `T[1, 2, 3]`은 1번 판, 2번 행, 3번 열의 값 23입니다(번호는 0부터 셉니다).

축 번호는 앞에서부터 0, 1, 2로 세고, 뒤에서부터 -1, -2, -3으로도 셉니다. `dim=-1`은 텐서가 몇 차원이든 늘 마지막 축이라, 트랜스포머 코드에서 가장 자주 보는 축 번호입니다.

- **인덱싱(indexing):** 축마다 번호를 골라 일부를 꺼내는 일입니다. 축 순서대로 쉼표로 나눠 적고, `:`는 그 축을 전부 고릅니다. 정수로 번호를 고르고 `:`로 축 전체를 고릅니다.
- **정수 하나로 고른 축은 결과에서 사라지고**, 범위(`1:`)로 고른 축은 크기가 줄어든 채 남습니다.
- **`...`:** 앞의 축을 모두 `:`로 채웁니다. `T[..., -1]`은 `T[:, :, -1]`과 같습니다.

<img src="images/72_tensor_axes.png" width="1050" alt="제목은 '축 번호: 앞에서 0, 1, 2 / 뒤에서 -3, -2, -1'입니다. 왼쪽 위에 '예시 텐서 T: 0~23을 (3, 4) 판 2장에 담음'과 T = torch.tensor(...) 코드가 적혀 있고, 첫 번째 판 [[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]은 보라 상자, 두 번째 판 [[12, 13, 14, 15], [16, 17, 18, 19], [20, 21, 22, 23]]은 자주 상자로 표시되어 범례 'T[0]: 첫 번째 판 (3, 4)', 'T[1]: 두 번째 판 (3, 4)'가 붙어 있습니다. 가운데 (2, 3, 4) 큐브는 앞면 0~11이 보라 테두리로 'T[0] = 앞 장', 윗면 12~15와 옆면 15, 19, 23이 자주 테두리로 'T[1] = 뒤 장'이고, 파랑 화살표 'dim=0 = dim=-3 (판 2장)', 초록 화살표 'dim=1 = dim=-2 (행 3개)', 주황 화살표 'dim=2 = dim=-1 (열 4개)'가 모서리를 따라 그려져 있습니다. 옆면의 23 칸이 빨갛게 강조되어 'T[1, 2, 3] = 23 (1번 판, 2번 행, 3번 열)'이 붙어 있습니다. 아래 '인덱싱으로 고르기' 판에는 작은 큐브 세 개로 'T[1] -> (3, 4) 뒤 장 전체', 'T[:, 0] -> (2, 4) 두 장의 0번 행', 'T[..., -1] -> (2, 3) 두 장의 마지막 열'이 고른 칸을 빨갛게 칠해 보입니다. 아래 띠에 'T[i]는 축 0에서 i번 판, T[i, j, k]는 i번 판, j번 행, k번 열의 값', 'dim=-1은 늘 마지막 축입니다'가 적혀 있습니다.">

In [ ]:
# 축 0에서 판 하나씩: (2, 3, 4) -> (3, 4)
print("T[0]", tuple(T[0].shape))
print(T[0])
print("T[1]", tuple(T[1].shape))
print(T[1])

# 축 세 개에 번호를 하나씩 주면 값 하나
print("T[1, 2, 3] =", T[1, 2, 3].item())

# 판마다 0번 행, 판마다 마지막 열
print("T[:, 0]   ", tuple(T[:, 0].shape), T[:, 0].tolist())
print("T[..., -1]", tuple(T[..., -1].shape), T[..., -1].tolist())

**결과 읽기**
- **`T[0]`은 0~11, `T[1]`은 12~23, 둘 다 (3, 4):** 축 0의 번호 하나를 고르면 판 한 장이 나오고 축 0은 사라집니다.
- **`T[1, 2, 3] = 23`:** 세 축 모두 번호를 고르면 숫자 하나입니다.
- **`T[:, 0]`은 (2, 4) [[0, 1, 2, 3], [12, 13, 14, 15]], `T[..., -1]`은 (2, 3) [[3, 7, 11], [15, 19, 23]]:** 번호를 고른 축만 사라지고 `:`로 둔 축은 남습니다.

## 3. (B, W_L, D): 문장 묶음을 담는 3차원 텐서

### 3-1. 토큰 ID에서 토큰 벡터로

교안 01은 리뷰 여러 개를 길이 W_L로 맞춘 토큰 ID 텐서 `(B, W_L)`로 만든 뒤, 토큰 임베딩으로 토큰 ID마다 벡터를 꺼내 `(B, W_L, D)` 텐서를 만듭니다. 같은 과정을 단어 8개짜리 어휘 사전과 문장 두 개로 따라갑니다. 보기 쉽게 단어 하나를 토큰 하나로 만든 예시라, 문자 토큰을 쓰는 교안 01과 토큰 ID는 다릅니다.

- **문장 0:** `[CLS] 영화 정말 재밌다 [PAD]`, **문장 1:** `[CLS] 너무 지루하다 [PAD] [PAD]`. 둘 다 W_L = 5로 맞췄습니다.
- **nn.Embedding(어휘 크기, D):** 토큰 ID마다 길이 D인 벡터를 한 행씩 담은 임베딩 행렬에서 그 토큰 ID의 행을 꺼냅니다. 숫자 하나(토큰 ID)가 숫자 D개로 바뀌므로 맨 뒤에 축 D가 하나 늘어납니다.

<img src="images/73_bwd_tensor.png" width="1050" alt="제목은 '(B, W_L, D): 문장 묶음이 3차원 텐서가 되기까지'입니다. 왼쪽 위 'tokens: (B, W_L) = (2, 5)' 표는 행 머리가 파랑 '문장 0', '문장 1', 열 머리가 초록 '위치 0'~'위치 4'이고, 칸마다 단어와 번호가 적혀 있습니다: [CLS] 1, 영화 2, 정말 3, 재밌다 4, [PAD] 0 / [CLS] 1, 너무 7, 지루하다 6, [PAD] 0, [PAD] 0. 오른쪽에 화살표 'embedding(tokens)'와 'padding_idx=0'이 있습니다. 그 아래 'embedding.weight (8, 4)' 표는 0 [PAD]부터 7 너무까지 8행, 주황 '성분 0'~'성분 3' 4열이고 0행은 모두 0이며, 빨간 테두리와 선이 토큰 번호에서 1 [CLS] (0.85, 0.69, -0.32, -2.12), 2 영화 (0.32, -1.26, 0.35, 0.31), 7 너무 (1.39, 1.59, 0.95, -0.84) 행으로 이어집니다. 오른쪽 위 'D (벡터 성분)' 판에는 '문장 0'과 '문장 1'의 5행 4열 격자 두 장이 값과 함께 있습니다. 문장 0은 [CLS], 영화, 정말, 재밌다 행에 위 값들이 들어 있고 [PAD] = 0 행은 모두 0이며, 문장 1은 [CLS], 너무, 지루하다 행 뒤에 [PAD] = 0 행이 두 개입니다. 오른쪽 아래에는 B=2, W_L=5, D=4가 표시된 작은 격자 큐브와 'h: (B, W_L, D) = (2, 5, 4)', '위의 두 격자는 한 큐브의 두 장', '문장 0 = 앞 장, 문장 1 = 뒤 장'이 적혀 있습니다. 아래 띠에 '토큰 ID (B, W_L) -> 토큰 벡터 (B, W_L, D)', '토큰 ID로 embedding.weight의 행을 꺼내 숫자 D개짜리 벡터로 바꿉니다', '이 예시 (2, 5) -> (2, 5, 4), 교안 01 (128, 64) -> (128, 64, 64)'가 적혀 있습니다.">

In [ ]:
# 단어 8개짜리 어휘 사전. 토큰 ID 0은 [PAD], 1은 [CLS]
vocab = ["[PAD]", "[CLS]", "영화", "정말", "재밌다", "별로", "지루하다", "너무"]
sentences = [["[CLS]", "영화", "정말", "재밌다", "[PAD]"],
             ["[CLS]", "너무", "지루하다", "[PAD]", "[PAD]"]]

# 단어를 토큰 ID로 바꿔 (B, W_L) 정수 텐서를 만듭니다
id_rows = []
for sentence in sentences:
    id_rows.append([vocab.index(word) for word in sentence])
tokens = torch.tensor(id_rows)
print("tokens", tuple(tokens.shape))
print(tokens)

# 토큰 ID마다 길이 4 벡터를 꺼내는 임베딩 층. [PAD](토큰 ID 0) 벡터는 0으로 둡니다
torch.manual_seed(0)
embedding = nn.Embedding(len(vocab), 4, padding_idx=0)

# 학습하지 않고 모양과 값만 보므로 기울기 기록을 끄고 계산합니다
with torch.no_grad():
    h = embedding(tokens)  # (B, W_L) -> (B, W_L, D)

B, W_L, D = h.shape
print(f"h {tuple(h.shape)}  ->  B = {B}, W_L = {W_L}, D = {D}")
print(h)

**결과 읽기**
- **tokens (2, 5) -> h (2, 5, 4):** 토큰 ID 10개가 각각 길이 4 벡터로 바뀌어, 숫자가 10개에서 40개로 늘었습니다.
- **두 문장의 0번 행이 같습니다 [0.85, 0.69, -0.32, -2.12]:** 둘 다 [CLS](토큰 ID 1)라서 임베딩 행렬의 같은 행을 꺼냈습니다. 인코더를 지나기 전에는 토큰 ID가 같으면 늘 같은 벡터입니다.
- **[PAD] 행은 모두 0:** `padding_idx=0`이 0번 행을 0으로 고정했습니다.

### 3-2. 인덱싱으로 문장, 위치, 범위 꺼내기

2-2절의 인덱싱 규칙을 `(B, W_L, D)`에 그대로 씁니다. 교안 01의 `cls_vector = h[:, 0]`이 문장마다 0번 위치 벡터를 꺼내는 줄입니다.

<img src="images/74_indexing_cls.png" width="1050" alt="제목은 '인덱싱: 3차원 텐서에서 원하는 부분 꺼내기'입니다. 세 판마다 오른쪽 위에 h (2, 5, 4)를 나타내는 작은 큐브가 있고, 그 아래 h의 '문장 0', '문장 1' 5행 4열 격자 두 장이 그려져 있으며 꺼낸 칸이 빨간 테두리로 강조되고, 화살표 아래에 꺼낸 결과가 있습니다. 첫 판 'h[0]' '문장 0 전체'는 문장 0 격자 전체가 빨간 테두리이고 '결과 (5, 4)', 'B 축이 사라짐'입니다. 둘째 판 'h[:, 0]' '문장마다 0번 위치 = [CLS] 벡터'는 두 문장의 맨 윗행이 빨간 테두리이고 '결과 (2, 4)', 'W_L 축이 사라짐', '두 문장 모두 [CLS]라 두 행의 값이 같음'입니다. 셋째 판 'h[:, 1:]' '[CLS]를 뺀 나머지 위치'는 두 문장의 윗행을 뺀 4행이 빨간 테두리이고 '결과 (2, 4, 4)', 'W_L 축이 5 -> 4로 줄어듦'입니다. 아래 띠에 '정수 하나로 고른 축은 사라지고(h[:, 0]), 범위(1:)로 고른 축은 남습니다', '코드 옆 작은 큐브 = h (2, 5, 4), 빨간 칸이 고른 부분 (앞 장 = 문장 0)', '교안 01: cls_vector = h[:, 0], 순위 게임 h[:, 1:]'가 적혀 있습니다.">

In [ ]:
# 문장 0 하나: B 축이 사라집니다
print("h[0]     ", tuple(h[0].shape))

# 문장마다 0번 위치([CLS]) 벡터: W_L 축이 사라집니다
print("h[:, 0]  ", tuple(h[:, 0].shape))
print(h[:, 0])

# [CLS]를 뺀 나머지 위치: W_L 축이 5에서 4로 줄어든 채 남습니다
print("h[:, 1:] ", tuple(h[:, 1:].shape))

**결과 읽기**
- **`h[0]`은 (5, 4):** 문장 하나의 토큰 벡터 5개입니다.
- **`h[:, 0]`은 (2, 4):** 리뷰 하나를 벡터 하나로 나타낸 `(B, D)`입니다. 교안 01에서는 이 벡터가 인코더를 지난 뒤 분류층으로 들어갑니다.
- **`h[:, 1:]`은 (2, 4, 4):** 범위로 고른 축은 남으므로 3차원 그대로입니다.

## 4. 내적과 행렬 곱

### 4-1. 내적

어텐션은 토큰끼리 얼마나 관련 있는지를 점수로 매기는데, 그 점수가 두 벡터의 내적입니다.

**내적(dot product)**
- 길이가 같은 두 벡터에서 같은 번호의 성분끼리 곱한 뒤 모두 더한 숫자 하나입니다.

$\displaystyle a \cdot b = a_0 b_0 + a_1 b_1 + \cdots + a_{D-1} b_{D-1}$

- 길이가 비슷하다면 두 벡터가 가리키는 방향이 비슷할수록 크고, 직각이면 0, 반대 방향이면 음수입니다. 그래서 내적을 두 벡터가 닮은 정도의 점수로 씁니다.
- 벡터가 길수록(성분 값이 클수록) 내적의 크기도 커집니다. 6-3절에서 점수를 나누는 이유가 이것입니다.

<img src="images/75_dot_product.png" width="1050" alt="제목은 '내적: 같은 번호의 성분끼리 곱해서 모두 더한 값'입니다. 왼쪽 판 '같은 번호의 성분끼리 곱해서 더하기'에는 '영화' [1, 0, 1, 0]과 '재밌다' [1, 1, 1, 0]이 성분 0~3 칸으로 위아래에 놓이고, 칸끼리 곱한 '곱' [1, 0, 1, 0]이 '더하기'를 지나 큰 상자 2가 됩니다. 그 아래 '1*1 + 0*1 + 1*1 + 0*0 = 2'와 '두 벡터가 함께 1인 성분이 많을수록 큼'이 적혀 있습니다. 오른쪽 판 '방향과 내적'은 좌표 평면에 a = (2, 1)과 세 화살표를 그렸습니다: '같은 쪽 (2, 2) 내적 6', '직각 (-1, 2) 내적 0', '반대쪽 (-2, -1) 내적 -5'. 아래 띠에 '내적 = a0*b0 + a1*b1 + ... (같은 번호의 성분끼리 곱해 더함)', '길이가 비슷하면 방향이 비슷할수록 크고, 직각이면 0, 반대면 음수', '어텐션은 내적으로 토큰끼리 얼마나 닮았는지 점수를 냅니다'가 적혀 있습니다.">

그림 왼쪽의 '영화'와 '재밌다' 벡터로 계산합니다. 단계는 두 개입니다.
1. 같은 번호의 성분끼리 곱합니다: `[1*1, 0*1, 1*1, 0*0] = [1, 0, 1, 0]`
2. 모두 더합니다: `1 + 0 + 1 + 0 = 2`

In [ ]:
# '영화'와 '재밌다'를 나타내는 벡터(예시 값)
a = torch.tensor([1.0, 0.0, 1.0, 0.0])
b = torch.tensor([1.0, 1.0, 1.0, 0.0])

# 1. 같은 번호의 성분끼리 곱하기(*), 2. 모두 더하기
products = a * b
print("a * b         ", products)
print("products.sum()", products.sum())

# 벡터끼리 @ 는 같은 계산(내적)을 한 번에 합니다
print("a @ b         ", a @ b)

# 방향에 따라 내적이 어떻게 달라지는지 길이 2 벡터(좌표 평면의 화살표)로 확인합니다
base = torch.tensor([2.0, 1.0])
for name, other in [("같은 쪽", torch.tensor([2.0, 2.0])),
                    ("직각", torch.tensor([-1.0, 2.0])),
                    ("반대쪽", torch.tensor([-2.0, -1.0]))]:
    print(f"{name}: (2, 1) · {other.tolist()} = {(base @ other).item()}")

**결과 읽기**
- **`a * b`는 [1, 0, 1, 0], 합 2:** `*`는 같은 번호의 성분끼리 곱하기만 하고 모양을 그대로 둡니다(원소별 곱, element-wise product). 더하기까지 해야 내적입니다.
- **`a @ b`도 2:** 1차원 벡터끼리 `@`를 쓰면 내적 하나(0차원 텐서)가 나옵니다.
- **같은 쪽 6, 직각 0, 반대쪽 -5:** 세 벡터의 길이가 비슷할 때는 방향이 비슷할수록 내적이 큽니다.

### 4-2. 행렬 곱: 칸마다 내적 하나

**행렬 곱(matrix multiplication)**
- `(n, k) @ (k, m)`의 결과는 `(n, m)`이고, 결과의 칸 `(i, j)`는 **앞 행렬의 i번 행과 뒤 행렬의 j번 열의 내적**입니다.
- 행과 열의 길이가 같아야 내적을 하므로, 앞 행렬의 마지막 크기와 뒤 행렬의 첫 크기(안쪽 크기 k)가 같아야 합니다. 결과 모양은 바깥 크기 `(n, m)`입니다.

<img src="images/76_matmul_dot.png" width="1050" alt="제목은 '행렬 곱 = 행과 열의 내적을 칸마다 하나씩'입니다. 맨 위에 모양 'X (2, 3) @ W (3, 4) -> X @ W (2, 4)'가 있고 안쪽 크기 3과 3이 빨간 원으로 표시되어 있습니다. 그 아래 X [[1, 2, 3], [4, 5, 6]] '@' W [[1, 0, 2, 0], [0, 1, 1, 2], [1, 1, 0, 1]] '=' 결과 [[4, 5, 4, 7], [10, 11, 13, 16]]이 '0번 행', '1번 행', '0번 열'~'3번 열' 이름과 함께 나란히 있습니다. 'X의 1번 행', 'W의 2번 열', '칸 (1, 2)'의 13이 빨간 테두리로 강조되고, 아래 상자에 '4*2 + 5*1 + 6*0 = 13'이 적혀 있습니다. 아래 띠에 '결과 칸 (i, j) = X의 i번 행과 W의 j번 열의 내적', '안쪽 크기가 같아야 곱할 수 있고(3 = 3), 결과 모양은 바깥 크기 (2, 4)'가 적혀 있습니다.">

칸 여덟 개를 차례로 채우는 순서입니다.

<img src="images/90_matmul_dot_animation.gif" width="900" alt="움직이는 그림. 제목은 '행렬 곱: 칸마다 행과 열의 내적'입니다. 첫 장에는 X (2, 3), W (3, 4), 빈 XW (2, 4)가 0번 행, 0번 열 같은 번호와 함께 놓이고 'XW[i, j] = X의 i번 행과 W의 j번 열을 내적한 값', 아래 띠에 'X (2, 3) @ W (3, 4): 결과 XW는 (2, 4)', '빈 칸 8개를 왼쪽 위부터 하나씩 채웁니다'가 적혀 있습니다. 다음 장마다 X의 한 행과 W의 한 열이 빨간 테두리로 강조되고, X[1, :], W[:, 0], 곱 표와 '합 = 10'처럼 계산이 보이며 결과 칸이 하나씩 채워집니다. 아래 띠에는 '1번 행 · 0번 열 = 4*1 + 5*0 + 6*1 = 10', 'XW[1, 0] = 10'처럼 그 칸의 식이 적힙니다. 마지막 장은 XW [[4, 5, 4, 7], [10, 11, 13, 16]] 전체와 '(2, 3) @ (3, 4) -> (2, 4)', '안쪽 3과 3이 같아야 곱할 수 있고, 남는 바깥 2와 4가 결과 모양', 아래 띠 '안쪽 크기 3 = 3, 결과 모양은 바깥 크기 (2, 4)'입니다.">

결과의 1번 행, 2번 열(번호는 0부터)을 손으로 계산하면 X의 1번 행 `[4, 5, 6]`과 W의 2번 열 `[2, 1, 0]`의 내적 `4*2 + 5*1 + 6*0 = 13`입니다.

In [ ]:
X = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])           # (2, 3)
W = torch.tensor([[1.0, 0.0, 2.0, 0.0],
                  [0.0, 1.0, 1.0, 2.0],
                  [1.0, 1.0, 0.0, 1.0]])      # (3, 4)

# 행렬 곱: (2, 3) @ (3, 4) -> (2, 4)
XW = X @ W
print("X @ W", tuple(XW.shape))
print(XW)

# 같은 결과를 칸마다 내적 하나씩 계산해 채웁니다
by_dot = torch.zeros(2, 4)
for i in range(2):
    for j in range(4):
        by_dot[i, j] = X[i] @ W[:, j]  # X의 i번 행 · W의 j번 열
print("칸마다 내적으로 채운 결과와 같은가:", torch.equal(by_dot, XW))
print("1번 행, 2번 열:", X[1].tolist(), "·", W[:, 2].tolist(), "=", (X[1] @ W[:, 2]).item())

**결과 읽기**
- **(2, 3) @ (3, 4) -> (2, 4), 값 [[4, 5, 4, 7], [10, 11, 13, 16]]:** 안쪽 크기 3이 사라지고 바깥 크기 2와 4가 남았습니다.
- **칸마다 내적으로 채운 결과와 같은가: True:** 행렬 곱은 내적 8개를 한 번에 계산한 것입니다.
- **곱할 수 없는 경우:** `W @ X`는 `(3, 4) @ (2, 3)`이라 안쪽 크기가 4와 2로 달라 `RuntimeError`가 납니다. 에러 메시지에 두 모양이 나오므로 안쪽 크기부터 확인합니다.

### 4-3. 3차원 텐서와 nn.Linear

`(B, W_L, D) @ (D, 3)`처럼 3차원 텐서에 2차원 행렬을 곱하면 **마지막 축에만** 행렬 곱을 하고 앞의 축은 그대로 둡니다. `nn.Linear`가 이 계산이라, `(B, W_L, D)`를 `nn.Linear(D, 3)`에 넣으면 토큰 벡터 B * W_L개가 각각 같은 가중치로 바뀌어 `(B, W_L, 3)`이 됩니다. 3차원 텐서끼리 곱하는 경우는 5-3절에서 봅니다.

- **nn.Linear(in, out)의 계산:** `x @ weight.T + bias`입니다. `weight`는 `(out, in)` 모양으로 저장되어 있어 행과 열을 바꿔(`.T`, 5-1절의 전치) 곱합니다. bias `(3,)`는 모든 토큰 벡터에 똑같이 더합니다.

<img src="images/79_linear_last_axis.png" width="1050" alt="제목은 'nn.Linear는 마지막 축만 바꿉니다: (B, W_L, D) -> (B, W_L, 3)'입니다. 왼쪽 'h (2, 5, 4)'는 문장 0, 문장 1의 위치 0~4, 성분 0~3 판 두 장, 가운데 'linear.weight.T (4, 3)'은 성분 0~3 행, 출력 0~2 열의 표와 '+ linear.bias (3,)' [0.18, -0.19, -0.04], 오른쪽 'out (2, 5, 3)'은 출력 0~2 열의 판 두 장이며 [PAD] 행은 '[PAD] = bias'로 0.18, -0.19, -0.04입니다. 문장 0의 위치 1(영화) 행과 그 결과 행이 빨간 테두리로 강조되고, 가운데 상자에 'h[0, 1] = [0.32, -1.26, 0.35, 0.31] (4개)', 'h[0, 1] @ linear.weight.T + linear.bias = [0.58, -0.68, 0.12] (3개)'가 적혀 있습니다. 아래 띠에 '토큰 벡터(행) 하나하나에 같은 weight.T를 곱합니다', '앞의 축 (B, W_L)은 그대로, 마지막 축만 D -> 3', '교안 01: W_Q, W_K, W_V, classifier 모두 nn.Linear'가 적혀 있습니다.">

In [ ]:
# 토큰 벡터(길이 D = 4)를 길이 3 벡터로 바꾸는 선형층
torch.manual_seed(1)
linear = nn.Linear(4, 3)
print("weight", tuple(linear.weight.shape), "bias", tuple(linear.bias.shape))

with torch.no_grad():
    out = linear(h)                                 # (2, 5, 4) -> (2, 5, 3)
    by_matmul = h @ linear.weight.T + linear.bias   # (2, 5, 4) @ (4, 3) + (3,) -> (2, 5, 3)

print("linear(h)", tuple(out.shape))
print("h @ weight.T + bias와 같은가:", torch.allclose(out, by_matmul))

# 토큰 하나(문장 0, 1번 위치)만 따로 넣어도 같은 행이 나옵니다
with torch.no_grad():
    one_token = linear(h[0, 1])                     # (4,) -> (3,)
print("문장 0, 1번 위치만 넣은 결과:", one_token)
print("linear(h)의 같은 위치      :", out[0, 1])

**결과 읽기**
- **(2, 5, 4) -> (2, 5, 3):** 마지막 축 D만 4에서 3으로 바뀌고 B, W_L은 그대로입니다.
- **같은가: True:** `nn.Linear`는 `@` 한 번과 bias 더하기입니다.
- **토큰 하나만 넣어도 같은 행:** 토큰마다 따로 계산하므로 다른 토큰의 값이 섞이지 않습니다. 토큰끼리 정보를 섞는 일은 어텐션이 맡습니다.

## 5. transpose: 두 축 바꾸기

어텐션은 문장 안 모든 토큰 쌍의 내적 점수가 필요합니다. 토큰 벡터를 행으로 쌓은 행렬끼리는 안쪽 크기가 맞지 않아 바로 곱할 수 없어, 한쪽의 행과 열을 바꾼 뒤 곱합니다.

### 5-1. 행렬의 전치

**전치(transpose)**
- 고른 두 축의 순서를 바꾸는 연산입니다. 행렬이면 행과 열이 바뀌어, 값 `M[i, j]`가 `M.T[j, i]`로 옮겨 갑니다.
- `M.T`와 `M.transpose(0, 1)`은 같은 결과입니다. 3차원 이상에서는 `transpose(축 a, 축 b)`로 바꿀 두 축을 적습니다.

<img src="images/80_transpose.png" width="1050" alt="제목은 'transpose: 두 축의 순서를 바꿉니다'입니다. 왼쪽 판 '2차원 표: 행이 열이 됩니다'는 'M.T  =  M.transpose(0, 1)' 아래 'M  (3, 4)' 0~11 표가 화살표를 지나 'M.T  (4, 3)'이 되는 그림입니다. 행마다 분홍, 노랑, 하늘색으로 칠해 M의 행이 M.T의 열이 되는 것을 보이고, 점선 대각선과 빨간 칸 6이 표시되어 있으며, 범례에 'M의 0번 행 (0 1 2 3) -> M.T의 0번 열', 'M의 1번 행 (4 5 6 7) -> M.T의 1번 열', 'M의 2번 행 (8 9 10 11) -> M.T의 2번 열', '점선(대각선) 위 값 0, 5, 10은 제자리', '빨간 칸: M[1, 2] = 6 -> M.T[2, 1] = 6'이 적혀 있습니다. 오른쪽 판 '3차원: 고른 두 축만 바꿉니다'는 격자 큐브 네 개로 위쪽 'h.transpose(-2, -1):  (2, 5, 4)  ->  (2, 4, 5)'에서 세로 W_L = 5와 가로 D = 4가 자리를 바꿔 세로 D = 4, 가로 W_L = 5가 되고 깊이 B = 2는 그대로이며, 'B(파랑)는 그대로, W_L(초록)과 D(주황)만 서로 바뀜'이 적혀 있습니다. 아래쪽 'h.transpose(0, 1):  (2, 5, 4)  ->  (5, 2, 4)'에서는 B = 2와 W_L = 5가 서로 바뀌어 깊이가 W_L = 5, 세로가 B = 2가 되고 D = 4는 그대로입니다. 아래 띠에 'transpose(a, b): 축 a와 축 b만 바꾸고 나머지 축은 그대로', '값 M[i, j]가 M.T[j, i] 칸으로 옮겨 갑니다'가 적혀 있습니다.">

<img src="images/93_transpose_animation.gif" width="900" alt="움직이는 그림. 제목은 'transpose: M[i, j] -> M.T[j, i]'입니다. 첫 장에는 M (3, 4) 0~11과 빈 M.T (4, 3)이 0번 행, 0번 열 같은 번호와 함께 놓이고 'M의 행 하나가 M.T의 열 하나가 됩니다', 아래 띠에 'M (3, 4)를 transpose하면 M.T (4, 3)', '행을 하나씩 옮겨 봅니다'가 적혀 있습니다. 다음 장마다 M의 한 행이 빨간 테두리로 강조되고 화살표를 따라 M.T의 같은 번호 열로 옮겨 채워지며, 아래 띠에 'M의 1번 행 (4, 5, 6, 7) -> M.T의 1번 열', 'M[1, j] = M.T[j, 1]'처럼 적힙니다. 마지막 장은 M.T [[0, 4, 8], [1, 5, 9], [2, 6, 10], [3, 7, 11]] 전체에서 M[1, 2]와 M.T[2, 1] 칸 6이 강조되고 'M[1, 2] = 6 = M.T[2, 1]', 아래 띠 '행이 열이 되고 모양은 (3, 4) -> (4, 3)', '예: M[1, 2] = 6은 M.T[2, 1] 칸으로 갑니다'입니다.">

In [ ]:
print("M", tuple(M.shape))
print(M)

# 행과 열을 바꿉니다: (3, 4) -> (4, 3)
print("M.T", tuple(M.T.shape))
print(M.T)
print("M.transpose(0, 1)과 같은가:", torch.equal(M.T, M.transpose(0, 1)))

# M의 1번 행, 2번 열 값이 M.T에서는 2번 행, 1번 열에 있습니다
print("M[1, 2] =", M[1, 2].item(), "| M.T[2, 1] =", M.T[2, 1].item())

**결과 읽기**
- **(3, 4) -> (4, 3):** M의 0번 행 [0, 1, 2, 3]이 M.T의 0번 열이 되었습니다.
- **M[1, 2] = 6, M.T[2, 1] = 6:** 값은 그대로이고 행 번호와 열 번호만 뒤바뀌었습니다.

### 5-2. 토큰끼리 내적: 점수 행렬

토큰 벡터들을 행으로 쌓은 행렬 `Q (토큰 수, D)`에 그 전치 `Q.T (D, 토큰 수)`를 곱하면, 모든 토큰 쌍의 내적이 한 번에 나옵니다. 전치해야 Q의 행(토큰 벡터)과 Q.T의 열(역시 토큰 벡터)이 만나 4-2절의 행렬 곱 규칙대로 칸마다 내적이 됩니다. 결과 `(토큰 수, 토큰 수)`의 i번 행, j번 열은 i번 토큰과 j번 토큰의 내적입니다.

- **Query와 Key:** 행의 토큰이 Query(점수를 매기는 쪽), 열의 토큰이 Key(점수를 받는 쪽)입니다.

<img src="images/77_scores_matmul.png" width="1050" alt="제목은 '토큰끼리 내적하면 점수 행렬이 됩니다: Q @ Q.T'입니다. 왼쪽부터 행 이름 영화, 정말, 재밌다와 성분 0~3의 Q (3, 4) = [[1, 0, 1, 0], [0, 1, 0, 1], [1, 1, 1, 0]], '@', 열 이름 영화, 정말, 재밌다의 Q.T (4, 3), '=', scores = Q @ Q.T (3, 3) = [[2, 0, 2], [0, 2, 1], [2, 1, 3]]이 파랑 농도로 칠해져 있습니다. Q의 '재밌다' 행, Q.T의 '영화' 열, scores의 (재밌다, 영화) 칸 2가 빨간 테두리로 강조되고, 대각선 칸은 점선 테두리이며 '점선 = 대각선 = 자기 자신과의 내적'이 적혀 있습니다. 가운데 상자에 '재밌다 · 영화 = 1*1 + 1*0 + 1*1 + 0*0 = 2'가 있습니다. 아래 띠에 '점수 행렬 (토큰 수, 토큰 수): 행 = Query(점수를 매기는 토큰), 열 = Key(점수를 받는 토큰)', '어텐션 점수 Q @ K.transpose(-2, -1)도 같은 계산입니다'가 적혀 있습니다.">

<img src="images/91_scores_matmul_animation.gif" width="900" alt="움직이는 그림. 제목은 '점수 행렬: 토큰끼리 내적 Q @ Q.T'입니다. 첫 장에는 Q (3, 4), Q.T (4, 3), 빈 scores (3, 3)이 행과 열 이름 영화, 정말, 재밌다와 함께 놓이고 'scores[i, j] = i번 토큰 벡터 · j번 토큰 벡터', 아래 띠에 '토큰 3개(영화, 정말, 재밌다)의 벡터 Q (3, 4)', 'Q @ Q.T로 모든 토큰 쌍의 내적을 한 번에 구합니다'가 적혀 있습니다. 다음 장마다 Q의 한 행과 Q.T의 한 열이 빨간 테두리로 강조되고, 두 벡터와 곱, '합 = 2' 같은 계산이 보이며 scores 칸이 하나씩 파랑으로 채워집니다. 아래 띠에는 '정말 · 정말 = 0*0 + 1*1 + 0*0 + 1*1 = 2', 'scores[1, 1] = 2 (정말 행, 정말 열)'처럼 그 칸의 식이 적힙니다. 마지막 장은 scores [[2, 0, 2], [0, 2, 1], [2, 1, 3]] 전체와 '(3, 4) @ (4, 3) -> (3, 3)', '점선 테두리: 대각선(같은 토큰끼리)', 아래 띠 '대각선은 자기 자신과의 내적', '행 = Query(점수를 매기는 토큰), 열 = Key(점수를 받는 토큰)'입니다.">

In [ ]:
# 토큰 3개의 벡터(예시 값)를 행으로 쌓습니다: (토큰 수, D) = (3, 4)
words = ["영화", "정말", "재밌다"]
Q = torch.tensor([[1.0, 0.0, 1.0, 0.0],
                  [0.0, 1.0, 0.0, 1.0],
                  [1.0, 1.0, 1.0, 0.0]])

# 모든 토큰 쌍의 내적: (3, 4) @ (4, 3) -> (3, 3)
scores = Q @ Q.T
print("scores", tuple(scores.shape))

# 행과 열에 토큰 이름을 붙여 표로 봅니다(행 = Query, 열 = Key)
display(pd.DataFrame(scores.numpy(), index=words, columns=words))

**결과 읽기**
- **(재밌다, 영화) = 2:** 4-1절에서 손으로 구한 내적과 같은 값이 표의 한 칸에 들어 있습니다.
- **(영화, 정말) = 0:** 두 벡터는 1이 있는 성분 번호가 겹치지 않아 내적이 0입니다.
- **표가 대각선을 기준으로 대칭입니다:** 같은 행렬 Q끼리 곱해 (i, j)와 (j, i)가 같은 내적입니다. 교안 01의 어텐션은 Query와 Key를 서로 다른 `nn.Linear`로 만들어 대칭이 아닙니다.

### 5-3. 어텐션 점수: Q @ K.transpose(-2, -1)

교안 01의 어텐션 점수는 `(B, W_L, D)` 모양의 Q와 K로 5-2절의 `Q @ Q.T`를 문장마다 계산한 것입니다. 3차원에서는 `.T` 대신 **마지막 두 축만** 바꾸는 `transpose(-2, -1)`을 씁니다.

- **배치 행렬 곱(batched matrix multiplication):** `@`는 3차원 이상 텐서를 받으면 마지막 두 축으로 행렬 곱을 하고, 앞의 축(B)은 같은 번호끼리 짝지어 따로 계산합니다.
- `(B, W_L, D) @ (B, D, W_L) -> (B, W_L, W_L)`: 문장마다 토큰 쌍의 점수 표가 하나씩 나옵니다.

<img src="images/81_batch_matmul.png" width="1050" alt="제목은 '배치 행렬 곱: 문장마다 따로 곱합니다 (B, W_L, D) @ (B, D, W_L)'입니다. 위에 'h @ h.transpose(-2, -1):  (2, 5, 4)  @  (2, 4, 5)  ->  (2, 5, 5)'가 축 색으로 적혀 있고, 그 아래 격자 큐브 세 개가 '@'와 '='로 이어져 각각 '(2, 5, 4)', '(2, 4, 5)', '(2, 5, 5)'로 표시되어 있습니다. 오른쪽에 '큐브의 앞 장 = 문장 0, 뒤 장 = 문장 1', '장마다 따로 (5, 4) @ (4, 5) = (5, 5)', '아래가 장 하나씩 펼친 계산입니다'가 적혀 있습니다. 그 아래 '문장 0 (앞 장)'과 '문장 1 (뒤 장)' 두 판마다 'h[b] (5, 4)', 'h[b].T (4, 5)', 'attn_scores[b] (5, 5)'가 그려져 있습니다. 점수 칸은 파랑 농도와 소수 둘째 자리 값이고 (문장 0 대각선 5.77, 1.92, 2.85, 5.66, 문장 1 대각선 5.77, 6.05, 0.97) 행과 열 이름은 문장 0이 [CLS], 영화, 정말, 재밌다, [PAD], 문장 1이 [CLS], 너무, 지루하다, [PAD], [PAD]입니다. [PAD] 행과 열은 회색 0이고, 판 오른쪽에 '[PAD] 행과 열은 벡터가 0이라 점수도 0', '칸 (i, j) = 토큰 i와 j의 내적'이 적혀 있습니다. 아래 띠에 '@는 마지막 두 축으로 행렬 곱을 하고, 앞의 축(B)은 문장마다 짝지어 따로 계산합니다', '그래서 Q @ K.transpose(-2, -1): (B, W_L, D) @ (B, D, W_L) -> (B, W_L, W_L)'가 적혀 있습니다.">

In [ ]:
# 모양만 보기 위해 Q와 K 자리에 h를 그대로 씁니다(교안 01에서는 W_Q, W_K를 지난 값이고 모양은 같습니다)
Q_batch = h   # (B, W_L, D)
K_batch = h   # (B, W_L, D)

# 마지막 두 축(W_L, D)만 바꿉니다. B 축은 그대로입니다
K_t = K_batch.transpose(-2, -1)
print("K.transpose(-2, -1)", tuple(K_t.shape))

# 문장마다 (W_L, D) @ (D, W_L): (2, 5, 4) @ (2, 4, 5) -> (2, 5, 5)
attn_scores = Q_batch @ K_t
print("Q @ K.transpose(-2, -1)", tuple(attn_scores.shape))

# 문장 1의 점수 표는 문장 1만 따로 계산한 것과 같습니다
print("문장 1만 따로 곱한 것과 같은가:", torch.allclose(attn_scores[1], h[1] @ h[1].T))

**결과 읽기**
- **(2, 5, 4) -> K.transpose (2, 4, 5) -> 점수 (2, 5, 5):** 문장마다 5 * 5 점수 표가 하나씩입니다.
- **같은가: True:** 배치 행렬 곱은 문장마다 5-2절의 계산을 따로 한 것입니다. 문장끼리는 값이 섞이지 않습니다.
- **transpose를 빠뜨리면:** `Q @ K`는 `(2, 5, 4) @ (2, 5, 4)`라 안쪽 크기 4와 5가 달라 `RuntimeError`가 납니다. 다만 교안 01처럼 W_L = D = 64이면 **에러 없이** 모양까지 같은 틀린 값이 나옵니다. 이 부록이 B, W_L, D를 모두 다른 크기(2, 5, 4)로 고른 이유입니다.

## 6. softmax: 점수를 합이 1인 비율로

### 6-1. softmax 계산 단계

내적 점수는 크기에 정해진 범위가 없어, 어텐션은 점수를 합이 1인 비율(어텐션 가중치)로 바꾼 뒤 그 비율로 토큰 벡터를 섞습니다(6-4절).

**softmax**
- 숫자 n개를 받아, 각 숫자에 지수 함수 $e^x$를 씌운 뒤 그 합으로 나눠 **0과 1 사이, 합이 1인 숫자 n개**를 만듭니다.

$\displaystyle \mathrm{softmax}(x)_i = \frac{e^{x_i}}{\sum_{j} e^{x_j}}$

- 점수의 순서는 그대로 두고, 큰 점수일수록 비율이 더 크게 벌어집니다. $e^x$가 모두 양수라 음수 점수도 0보다 큰 비율을 받습니다.

<img src="images/78_softmax.png" width="1050" alt="제목은 'softmax: 점수를 합이 1인 비율로 바꿉니다'입니다. 위쪽은 '재밌다 행'의 '점수' 영화 2, 정말 1, 재밌다 3이 'e^x'를 지나 7.39, 2.72, 20.09(합 = 30.19)가 되고, '합 30.19로 나누기'를 지나 '비율' 0.24, 0.09, 0.67(합 = 1.00)이 되는 흐름이며, 칸 아래 막대는 점수가 파랑, e^x가 회색, 비율이 주황입니다. 아래쪽은 'scores (3, 3)' [[2, 0, 2], [0, 2, 1], [2, 1, 3]](파랑)이 'torch.softmax(scores, dim=-1)', '행렬 전체를 한 번에, 행마다 따로'를 지나 'probs (3, 3): 비율' [[0.47, 0.06, 0.47], [0.09, 0.67, 0.24], [0.24, 0.09, 0.67]](주황)이 되고 행마다 '합 1.00'이 붙어 있습니다. 아래 띠에 'softmax(x)_i = e^(x_i) / (e^x 모두의 합)', '큰 점수일수록 큰 비율, 순서는 그대로, 모두 0과 1 사이', 'dim=-1: 마지막 축(행 안의 칸들)끼리 나눠 가지므로 행마다 합이 1입니다'가 적혀 있습니다.">

<img src="images/92_softmax_animation.gif" width="900" alt="움직이는 그림. 제목은 'softmax: 점수 -> 합이 1인 비율'입니다. 첫 장 위에 ''재밌다' 행: 영화, 정말, 재밌다와의 점수 scores[2]'가 있고 '점수' 막대 2, 1, 3만 그려져 있으며 'e^점수', '비율 = e^점수 / 합' 칸은 비어 있습니다. 다음 장들에서 'e^x'를 지나 e^점수 막대 7.39, 2.72, 20.09와 '합 = e^2 + e^1 + e^3 = 30.19'가, 이어서 '/ 합'을 지나 비율 막대 0.24, 0.09, 0.67과 '합 30.19로 나누면 비율 합 = 1.00'이 나타납니다. 그다음 장들에서는 scores (점수) [[2, 0, 2], [0, 2, 1], [2, 1, 3]]의 행이 하나씩 빨간 테두리로 강조되며 probs (비율)의 같은 행이 채워지고 '행 합' 1.00이 붙으며, 'dim=-1: 한 행(기준 토큰 하나) 안에서만 비율을 나눕니다'가 적혀 있습니다. 마지막 장은 probs [[0.47, 0.06, 0.47], [0.09, 0.67, 0.24], [0.24, 0.09, 0.67]] 전체와 아래 띠 '행마다 e^x / 합, 행 합 1.00', 'probs = softmax(scores, dim=-1)'입니다.">

5-2절 점수 표의 '재밌다' 행 `[2, 1, 3]`으로 세 단계를 따라갑니다.
1. 지수 함수: `[e^2, e^1, e^3] = [7.39, 2.72, 20.09]`
2. 합: `e^2 + e^1 + e^3 = 30.19`(반올림하기 전 값으로 더합니다)
3. 나누기: `[7.39 / 30.19, 2.72 / 30.19, 20.09 / 30.19] = [0.24, 0.09, 0.67]`

In [ ]:
# 5-2절 점수 표의 '재밌다' 행: [2, 1, 3]
row = scores[2]

# 1. 지수 함수, 2. 합, 3. 합으로 나누기
exp_row = torch.exp(row)
total = exp_row.sum()
probs_row = exp_row / total
print("점수        ", row)
print("e^점수      ", exp_row)
print("합          ", round(total.item(), 2))
print("softmax     ", probs_row, " 합:", round(probs_row.sum().item(), 2))

# torch.softmax로 한 번에 계산해도 같습니다
print("torch.softmax", torch.softmax(row, dim=-1))

# 점수가 1 큰 칸(3과 2)의 비율은 몇 배인지 반올림 전 값으로 계산합니다
print("비율 '재밌다' / '영화':", round((probs_row[2] / probs_row[0]).item(), 4))

**결과 읽기**
- **[2, 1, 3] -> [0.24, 0.09, 0.67]:** 가장 큰 점수 3(재밌다 자신)이 가장 큰 비율을 받고, 점수 순서(3 > 2 > 1)가 그대로 유지됩니다.
- **비율 '재밌다' / '영화' = 2.7183:** `e^3 / e^2 = e`라서 점수가 1 클 때마다 비율이 약 2.72배가 됩니다. 점수의 차이가 곱셈 배율로 바뀝니다.

### 6-2. 행렬에 softmax: dim으로 나눠 가질 칸 고르기

행렬에 softmax를 씌울 때는 어느 칸들끼리 합이 1이 되게 나눌지 `dim`으로 고릅니다. 어텐션은 **행마다**(Query 토큰마다) Key 토큰들에 비율을 나눠야 하므로 마지막 축 `dim=-1`을 씁니다.

- 합을 확인하는 `sum(dim=-1)`은 행마다 합 하나를, `sum(dim=0)`은 축 0(행 번호)을 따라 더해 열마다 합 하나를 냅니다.

<img src="images/99_softmax_dim.png" width="1050" alt="제목은 'softmax의 dim: 어느 칸끼리 합이 1인지 고릅니다'입니다. 왼쪽에 scores (3, 3) [[2, 0, 2], [0, 2, 1], [2, 1, 3]]이 파랑으로 있고, 오른쪽에 두 결과가 나란히 있습니다. 'torch.softmax(scores, dim=-1)' '행마다 합 1'은 [[0.47, 0.06, 0.47], [0.09, 0.67, 0.24], [0.24, 0.09, 0.67]]에 행마다 테두리가 쳐져 있고 '행 합' 1.00, 1.00, 1.00과 '한 행 안의 칸 3개가 1을 나눠 가짐'이 적혀 있습니다. 'torch.softmax(scores, dim=0)' '열마다 합 1'은 [[0.47, 0.09, 0.24], [0.06, 0.67, 0.09], [0.47, 0.24, 0.67]]에 열마다 테두리가 쳐져 있고 '열 합' 1.00, 1.00, 1.00, 빨간 글씨 '행 합' 0.80, 0.82, 1.38과 '1이 아님', '한 열 안의 칸 3개가 1을 나눠 가짐', '모양은 같은 (3, 3)'이 적혀 있습니다. 아래 띠에 'dim=-1: 행 안에서 나눠 가짐(어텐션은 Query 행마다)', 'dim=0도 모양은 (3, 3)이라 에러 없이 값만 틀립니다'가 적혀 있습니다.">

In [ ]:
# 행마다 나눠 갖기(dim=-1): Query 토큰마다 Key 토큰 3개에 비율을 나눕니다
probs = torch.softmax(scores, dim=-1)
display(pd.DataFrame(probs.numpy(), index=words, columns=words).round(2))
print("행마다 합(dim=-1로 더하기):", probs.sum(dim=-1))

# 열마다 나눠 갖기(dim=0)와 비교합니다
probs_col = torch.softmax(scores, dim=0)
print("dim=0으로 softmax 하면 열마다 합이 1:", probs_col.sum(dim=0))
print("이때 행마다 합:", probs_col.sum(dim=-1))

**결과 읽기**
- **`dim=-1`이면 행마다 합 [1.00, 1.00, 1.00]:** '영화' 행은 [0.47, 0.06, 0.47]로, 내적이 2로 같은 '영화'와 '재밌다'가 같은 비율을 받습니다.
- **`dim=0`이면 열마다 합이 1이고 행 합은 [0.80, 0.82, 1.38]:** 모양은 둘 다 (3, 3)이라 에러가 나지 않습니다. 축을 잘못 고르면 값만 조용히 틀립니다.

### 6-3. 점수의 크기와 softmax

softmax는 점수끼리의 **차이**로 비율을 정해, 점수에 같은 수를 더하면 결과가 그대로이고 점수를 나눠 차이를 줄이면 비율이 고르게 퍼집니다. 벡터가 길면 내적이 커져(4-1절) softmax가 한 칸에 거의 1을 몰아주므로, 어텐션은 점수를 $\sqrt{d_k}$로 나눠 차이를 줄입니다. $d_k$는 Key 벡터의 길이로, 이 부록에서는 D = 4입니다.

<img src="images/98_softmax_scale.png" width="1050" alt="제목은 '점수의 크기와 softmax: 차이가 비율을 정합니다'입니다. 네 판이 나란히 있고, 판마다 위에 점수 칸(파랑), 'softmax' 화살표, 아래에 영화, 정말, 재밌다의 주황 막대와 값이 있습니다. '원래 점수' scores [2, 1, 3] -> 0.24, 0.09, 0.67. '모두 10 더하기' scores + 10 [12, 11, 13] -> 0.24, 0.09, 0.67. '제곱근(d_k) = 2로 나누기' scores / 2 [1.00, 0.50, 1.50] -> 0.31, 0.19, 0.51. '3배 하기' scores * 3 [6, 3, 9] -> 0.05, 0.00, 0.95. 아래 띠에 '같은 수를 더하면 그대로, 나누면 고르게, 곱하면 한 칸으로 쏠립니다', '벡터의 성분이 많으면 내적이 커져서 한 칸으로 쏠립니다', '그래서 어텐션은 점수를 제곱근(d_k)로 나눕니다 (이 예시는 d_k = D = 4)'가 적혀 있습니다.">

In [ ]:
# '재밌다' 행으로 세 경우를 비교합니다
cases = {
    "원래 점수 [2, 1, 3]": row,
    "모두 10 더하기": row + 10,
    "제곱근(d_k) = 2로 나누기": row / D ** 0.5,  # 이 부록은 d_k = D = 4
    "3배 하기": row * 3,
}
case_rows = []
for name, values in cases.items():
    p = torch.softmax(values, dim=-1)
    case_rows.append({"점수": name, "영화": p[0].item(), "정말": p[1].item(), "재밌다": p[2].item()})
display(pd.DataFrame(case_rows).round(2))

**결과 읽기**
- **모두 10을 더해도 [0.24, 0.09, 0.67] 그대로:** 비율은 점수끼리의 차이로만 정해집니다.
- **2로 나누면 [0.31, 0.19, 0.51]:** 점수 차이가 줄어 비율이 고르게 퍼졌습니다.
- **3배 하면 [0.05, 0.00, 0.95]:** 점수 차이가 커지자 가장 큰 칸이 비율을 거의 다 가져갔습니다. 점수 1인 칸의 비율은 0.00으로 반올림될 만큼 작아졌습니다.

### 6-4. 비율로 벡터 섞기: 가중합

softmax로 만든 비율은 토큰 벡터를 섞는 데 씁니다. 비율 행렬 `(토큰 수, 토큰 수)`에 토큰 벡터 행렬 `(토큰 수, D)`를 곱하면, 결과의 i번 행은 **토큰 벡터들을 i번 행의 비율만큼 곱해 더한 벡터(가중합, weighted sum)** 입니다. 행렬 곱을 4-2절처럼 칸마다 내적으로 읽어도 같은 값이지만, 여기서는 행 단위로 읽는 편이 뜻이 잘 보입니다.

<img src="images/96_weighted_sum.png" width="1050" alt="제목은 '비율로 벡터 섞기: probs @ Q'입니다. 위에 probs (3, 3) [[0.47, 0.06, 0.47], [0.09, 0.67, 0.24], [0.24, 0.09, 0.67]](주황) '@' Q (3, 4) [[1, 0, 1, 0], [0, 1, 0, 1], [1, 1, 1, 0]] '=' probs @ Q (3, 4) [[0.94, 0.53, 0.94, 0.06], [0.33, 0.91, 0.33, 0.67], [0.91, 0.76, 0.91, 0.09]]가 행 이름 영화, 정말, 재밌다와 함께 놓이고, probs와 결과의 '재밌다' 행이 빨간 테두리로 강조되어 있습니다. 아래 ''재밌다' 행: 비율만큼 줄인 토큰 벡터를 더하기'에는 '0.24 * 영화 [1, 0, 1, 0]', '0.09 * 정말 [0, 1, 0, 1]', '0.67 * 재밌다 [1, 1, 1, 0]' 막대가 더해져 '결과의 '재밌다' 행 [0.91, 0.76, 0.91, 0.09]'가 되고, '점선 = 원래 벡터, 채운 막대 = 비율을 곱한 값'이 적혀 있습니다. 아래 띠에 '결과의 한 행 = 그 행의 비율로 섞은 토큰 벡터 하나', '(3, 3) @ (3, 4) -> (3, 4): 모양은 다시 (토큰 수, D)'가 적혀 있습니다.">

<img src="images/97_weighted_sum_animation.gif" width="900" alt="움직이는 그림. 제목은 '가중합: 비율만큼 곱해서 더하기'입니다. 첫 장에는 'probs[2]('재밌다' 행 비율)와 토큰 벡터 Q' 아래 영화 0.24, 정말 0.09, 재밌다 0.67과 벡터 막대 [1, 0, 1, 0], [0, 1, 0, 1], [1, 1, 1, 0]이, 오른쪽에 빈 '결과 벡터 = 곱한 벡터의 합'(0번 성분~3번 성분)이 놓이고, 아래 띠에 ''재밌다' 행의 비율로 세 토큰 벡터를 섞습니다', '영화 0.24, 정말 0.09, 재밌다 0.67'이 적혀 있습니다. 다음 세 장에서 영화, 정말, 재밌다 벡터가 차례로 빨간 테두리로 강조되어 '* 0.24'처럼 비율을 곱한 막대(점선은 원래 높이)가 결과 막대 위에 쌓이고, 아래 띠에 '영화 벡터 [1, 0, 1, 0] * 0.24 = [0.24, 0.00, 0.24, 0.00]', '곱한 벡터를 결과 막대에 쌓습니다'처럼 적힙니다. 마지막 장은 결과 막대 0.91, 0.76, 0.91, 0.09가 빨간 테두리로 묶이고 아래 띠에 '비율이 큰 토큰의 벡터를 많이 닮은 새 벡터: probs @ Q의 '재밌다' 행', '결과 [0.91, 0.76, 0.91, 0.09]'가 적힙니다.">

'재밌다' 행의 비율 `[0.24, 0.09, 0.67]`로 손으로 섞어 봅니다.
1. 비율만큼 곱합니다: `0.24 * [1, 0, 1, 0]`, `0.09 * [0, 1, 0, 1]`, `0.67 * [1, 1, 1, 0]`
2. 성분 번호별로 더합니다: `[0.24 + 0.67, 0.09 + 0.67, 0.24 + 0.67, 0.09] = [0.91, 0.76, 0.91, 0.09]`

In [ ]:
# 비율 (3, 3) @ 토큰 벡터 (3, 4) -> 섞은 벡터 (3, 4)
mixed = probs @ Q
print("probs @ Q", tuple(mixed.shape))
component_names = [f"성분 {k}" for k in range(D)]
display(pd.DataFrame(mixed.numpy(), index=words, columns=component_names).round(2))

# '재밌다' 행을 비율만큼 곱해 더하는 식으로 직접 계산해 비교합니다
by_weights = probs[2, 0] * Q[0] + probs[2, 1] * Q[1] + probs[2, 2] * Q[2]
print("'재밌다' 행 직접 계산:", by_weights)
print("probs @ Q의 같은 행과 같은가:", torch.allclose(by_weights, mixed[2]))

**결과 읽기**
- **(3, 3) @ (3, 4) -> (3, 4):** 토큰마다 섞은 벡터가 하나씩 나와, 모양이 다시 `(토큰 수, D)`입니다. 어텐션 출력이 입력과 같은 `(B, W_L, D)`인 이유입니다.
- **'재밌다' 행 [0.91, 0.76, 0.91, 0.09]:** 비율 0.67을 받은 '재밌다' 벡터 [1, 1, 1, 0]을 가장 많이 닮았고, 0.24를 받은 '영화'의 성분도 섞였습니다.
- **'영화' 행 [0.94, 0.53, 0.94, 0.06]:** 비율이 '영화'와 '재밌다'에 0.47씩 나뉘어, 두 벡터를 반씩 섞은 [1, 0.5, 1, 0]에 가깝습니다. 교안 01의 어텐션은 이렇게 섞을 대상으로 `W_V`를 지난 Value 벡터를 씁니다.

## 7. reshape: 같은 순서로 다시 자르기

원소의 순서는 그대로 두고 모양만 바꾸는 연산이 reshape입니다. 토큰 벡터의 성분을 여러 묶음으로 자를 때(교안 02의 직접 구현) 씁니다.

### 7-1. 원소 순서와 -1

**reshape**
- 원소를 **원래 순서 그대로** 한 줄로 읽어, 새 모양에 차례로 채웁니다. 읽고 채울 때는 마지막 축의 번호가 가장 빨리 바뀝니다(행 우선, row-major: 왼쪽에서 오른쪽, 그다음 아래 줄).
- 원소 수가 같아야 합니다. `(3, 4)`는 `(4, 3)`, `(2, 6)`, `(2, 2, 3)`으로 바꿀 수 있고 `(5, 2)`로는 못 바꿉니다.
- **-1:** 그 축의 크기를 남은 원소 수로 계산하라는 표시입니다. 원소 12개를 `reshape(4, -1)`하면 `-1`은 `12 / 4 = 3`이 됩니다. 한 번만 쓸 수 있습니다.

<img src="images/82_reshape_order.png" width="1050" alt="제목은 'reshape: 원소 순서는 그대로, 줄 바꿈 위치만 바뀝니다'입니다. 위쪽에 'flat = torch.arange(12)' (12,)의 0~11 열두 칸이 색 그라데이션으로 한 줄에 놓이고 '읽는 순서' 화살표와 'M.reshape(-1)도 (12,)로 폅니다'가 있습니다. 아래에 같은 색으로 채운 세 결과가 화살표로 이어져 있습니다: 'flat.reshape(3, 4)' -> (3, 4), 'flat.reshape(4, -1)' -> (4, 3), 'flat.reshape(2, 2, 3)' -> [0] 0~5, [1] 6~11의 (2, 2, 3). 가운데 아래에 '-1은 남은 크기를 자동 계산: 12 / 4 = 3'이 빨간 글씨로 적혀 있습니다. 아래 띠에 '원소 수가 같아야 합니다: 3 * 4 = 4 * 3 = 2 * 2 * 3 = 12', '읽는 순서: 마지막 축의 번호가 가장 빨리 바뀝니다(왼쪽에서 오른쪽, 그다음 아래 줄)'가 적혀 있습니다.">

<img src="images/94_reshape_animation.gif" width="900" alt="움직이는 그림. 제목은 'reshape: 순서대로 읽어 새 모양에 채웁니다'입니다. 첫 장에는 색 그라데이션의 M (3, 4) 0~11, 'reshape(4, 3)' 화살표, 빈 M.reshape(4, 3), 빈 '읽는 순서' 열두 칸이 놓이고, 아래 띠에 'M을 왼쪽 위부터 한 줄씩 읽어 0, 1, 2, ..., 11 순서로 늘어놓고', '그 순서 그대로 (4, 3)의 칸을 채웁니다'가 적혀 있습니다. 다음 장마다 M에서 세 칸이 빨간 테두리로 강조되어 읽는 순서 줄과 새 모양의 한 행에 채워지고, 아래 띠에 '읽은 순서 3 ~ 5번째 (3, 4, 5) -> 새 모양의 1번 행'처럼 적히며 '읽은 순서 9 ~ 11번째 (9, 10, 11) -> 새 모양의 3번 행'까지 이어집니다. 마지막 장은 'reshape(2, 2, 3)'으로 같은 순서를 M.reshape(2, 2, 3)의 [0] 0~5, [1] 6~11에 채운 그림과 아래 띠 '같은 순서 0 ~ 11을 (2, 2, 3)에 채우면 2 * 3 표가 2개', '원소 수 3 * 4 = 2 * 2 * 3 = 12가 같아야 reshape할 수 있습니다'입니다.">

In [ ]:
flat = torch.arange(12)
print("flat", tuple(flat.shape), flat)

# 같은 원소 12개를 여러 모양으로 다시 자릅니다
print("reshape(3, 4)", tuple(flat.reshape(3, 4).shape))
print(flat.reshape(3, 4))
print("reshape(4, 3)", tuple(flat.reshape(4, 3).shape))
print(flat.reshape(4, 3))
print("reshape(2, 2, 3)", tuple(flat.reshape(2, 2, 3).shape))
print(flat.reshape(2, 2, 3))

# -1은 남은 크기를 계산합니다
print("reshape(4, -1)", tuple(flat.reshape(4, -1).shape), "| M.reshape(-1)", tuple(M.reshape(-1).shape))

**결과 읽기**
- **(4, 3)의 0번 행은 [0, 1, 2], 1번 행은 [3, 4, 5]:** 0부터 차례로 세 개씩 끊어 채웠습니다. (2, 2, 3)도 판 2장에 같은 순서로 채워, 어느 모양이든 한 줄로 읽으면 0, 1, 2, ..., 11입니다.
- **reshape(4, -1)은 (4, 3), M.reshape(-1)은 (12,):** `-1` 축에 남은 크기가 들어갔습니다. `reshape(-1)`은 텐서를 한 줄로 펼칩니다.

### 7-2. reshape와 transpose는 다릅니다

`(3, 4)`를 `(4, 3)`으로 만드는 방법은 `reshape(4, 3)`과 `.T` 두 가지이고 값은 다릅니다. reshape는 순서를 지키며 다시 자르고, transpose는 행과 열의 뜻을 바꿉니다.

<img src="images/83_reshape_vs_transpose.png" width="1050" alt="제목은 '모양은 같아도 값이 다릅니다: reshape(4, 3)와 .T'입니다. 왼쪽 'M  (3, 4)'는 0~11이 적힌 표로 행마다 분홍, 노랑, 하늘색이 다르고 범례에 '0번 행', '1번 행', '2번 행', '색 = M에서 몇 번째 행이었나'가 있습니다. 가운데 'M.reshape(4, 3)'는 [[0, 1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11]]이고 오른쪽 'M.T'는 [[0, 4, 8], [1, 5, 9], [2, 6, 10], [3, 7, 11]]이며 값마다 M에서의 행 색을 그대로 씁니다. 두 판 위에 빨간 글씨 '둘 다 (4, 3)'과 괄호선, 두 판 사이에 빨간 글씨 '값 배치가 다릅니다'가 있습니다. 그 아래 'reshape(4, 3): 0 1 2 / 3 4 5 / 6 7 8 / 9 10 11', 'M.T: 0 4 8 / 1 5 9 / 2 6 10 / 3 7 11', 'reshape는 읽는 순서대로 다시 자르고, .T는 행을 열로 세웁니다'가 적혀 있습니다. 아래 띠에 '축의 뜻을 바꾸려면 transpose, 같은 순서로 다시 자르려면 reshape', '모양만 같다고 같은 텐서가 아닙니다. 값의 배치까지 확인합니다'가 적혀 있습니다.">

In [ ]:
# 둘 다 (4, 3)이지만 0번 행부터 다릅니다
reshaped = M.reshape(4, 3)
transposed = M.T
print("M.reshape(4, 3)", tuple(reshaped.shape), "0번 행:", reshaped[0])
print("M.T            ", tuple(transposed.shape), "0번 행:", transposed[0])
print("값이 같은가:", torch.equal(reshaped, transposed))

**결과 읽기**
- **둘 다 (4, 3)이고 값이 같은가: False:** `M.reshape(4, 3)`의 0번 행은 [0, 1, 2]로 M의 0번 행 앞부분이고, `M.T`의 0번 행은 [0, 4, 8]로 M의 0번 열입니다.
- 모양 주석이 원하는 모양이어도, 축의 뜻(어느 축이 토큰이고 어느 축이 성분인지)이 바뀌어야 하면 reshape가 아니라 transpose를 씁니다.

## 8. squeeze와 unsqueeze: 크기 1인 축

위치 번호를 세로 한 줄로 세우거나 출력 `(B, W_L, 1)`을 `(B, W_L)`로 펼 때처럼, 크기 1인 축을 넣고 빼서 모양을 맞추는 일이 자주 있습니다.

### 8-1. unsqueeze: 크기 1인 축 넣기

**unsqueeze(dim)**
- 값은 그대로 두고 `dim` 번호 위치에 크기 1인 축을 하나 끼워 넣습니다. `(4,)`에 `unsqueeze(0)`이면 `(1, 4)`, `unsqueeze(1)`이면 `(4, 1)`입니다.
- 원소 수가 그대로라 `reshape(1, 4)`와 같은 결과지만, 어느 위치에 축을 넣는지 코드에 바로 드러납니다.

<img src="images/84_unsqueeze_squeeze.png" width="1050" alt="제목은 'unsqueeze와 squeeze: 크기 1인 축을 넣고 뺍니다'입니다. 네 판입니다. 'unsqueeze(dim): dim 위치에 크기 1인 축 넣기' 판은 V (4,) [1, 2, 3, 4]가 'V.unsqueeze(0)'으로 '(1, 4) 가로 한 줄', 'V.unsqueeze(1)'로 '(4, 1) 세로 한 줄'이 되는 그림입니다. '교안 01: 위치 번호를 세로로 세우기' 판은 'torch.arange(5).unsqueeze(1)'로 arange (5,) [0, 1, 2, 3, 4]가 position_ids (5, 1)이 되고 'positional_encoding 함수의 첫 줄입니다'가 적혀 있습니다. 'squeeze(-1): 끝의 크기 1인 축 빼기' 판은 'token_scores.squeeze(-1)'로 얇은 (2, 5, 1) 큐브(깊이 2, 5행, 1열)가 (2, 5) 표가 되고 '교안 01 순위 게임의 로짓도 이렇게'가 적혀 있습니다. 빨간 테두리 '주의: 문장 하나(배치 1)에 squeeze()' 판은 얇은 (1, 5, 1) 큐브에 'one_sentence.squeeze()'를 하면 '(5,) 배치 축까지 사라짐', 'one_sentence.squeeze(-1)'이면 '(1, 5) 배치 축 그대로'입니다. 아래 띠에 '값은 그대로, 크기 1인 축만 더하거나 뺍니다', 'squeeze는 dim을 꼭 적습니다: 문장 하나(배치 1)일 때 배치 축까지 지워집니다'가 적혀 있습니다.">

In [ ]:
print("V", tuple(V.shape), V)
print("V.unsqueeze(0)", tuple(V.unsqueeze(0).shape))
print(V.unsqueeze(0))
print("V.unsqueeze(1)", tuple(V.unsqueeze(1).shape))
print(V.unsqueeze(1))

# 교안 01 위치 인코딩의 첫 줄: 위치 번호 0 ~ W_L - 1을 세로 한 줄로 세웁니다
position_ids = torch.arange(W_L).unsqueeze(1)
print("torch.arange(W_L).unsqueeze(1)", tuple(position_ids.shape))

**결과 읽기**
- **(4,) -> (1, 4)는 가로 한 줄, (4, 1)은 세로 한 줄:** 값 [1, 2, 3, 4]는 그대로이고 대괄호만 한 겹 늘었습니다.
- **position_ids (5, 1):** 위치 번호 0~4가 세로 한 줄로 섰습니다. 교안 01 `positional_encoding` 함수의 첫 줄이 이 계산입니다.

### 8-2. squeeze: 크기 1인 축 빼기

**squeeze(dim)**
- `dim` 번호의 크기 1인 축을 뺍니다. 그 축의 크기가 1이 아니면 에러 없이 모양을 그대로 둡니다. 교안 01 순위 게임은 숫자 위치마다 로짓(점수) 하나를 내는 `nn.Linear(d_model, 1)`의 출력 `(B, 20, 1)`을 `squeeze(-1)`로 `(B, 20)`으로 만듭니다.
- `dim` 없이 `squeeze()`를 쓰면 크기가 1인 축을 **모두** 뺍니다. 문장 하나만 넣어 B = 1이면 배치 축까지 사라집니다.

In [ ]:
# 토큰 위치마다 점수 하나: (B, W_L, D) -> (B, W_L, 1)
torch.manual_seed(0)
score_layer = nn.Linear(D, 1)
with torch.no_grad():
    token_scores = score_layer(h)
print("score_layer(h)", tuple(token_scores.shape), "-> squeeze(-1)", tuple(token_scores.squeeze(-1).shape))

# 문장 하나(B = 1)만 넣은 경우: squeeze()는 배치 축까지 지웁니다
with torch.no_grad():
    one_sentence = score_layer(h[:1])  # h[:1]은 범위로 골라 B 축이 남습니다: (1, 5, 4)
print("문장 하나", tuple(one_sentence.shape))
print("squeeze()  ", tuple(one_sentence.squeeze().shape))
print("squeeze(-1)", tuple(one_sentence.squeeze(-1).shape))

**결과 읽기**
- **(2, 5, 1) -> (2, 5):** 마지막 크기 1 축만 빠지고 값은 그대로입니다.
- **(1, 5, 1)에 `squeeze()`는 (5,), `squeeze(-1)`은 (1, 5):** dim을 적지 않으면 배치 축이 사라져, 뒤에서 `(B, ...)` 모양을 기대하는 코드가 틀어집니다. squeeze에는 늘 dim을 적습니다.

## 9. 핵심 정리

- **shape와 축:** shape는 축마다의 크기, `dim=-1`은 늘 마지막 축입니다. 정수로 인덱싱한 축은 사라지고(`h[:, 0]`: (2, 5, 4) -> (2, 4)) 범위로 고른 축은 남습니다(`h[:, 1:]`: (2, 4, 4)).
- **(B, W_L, D):** 토큰 ID `(B, W_L)`이 `nn.Embedding`으로 `(B, W_L, D)`가 됩니다. 이 부록은 (2, 5, 4), 교안 01은 (128, 64, 64)입니다.
- **내적과 행렬 곱:** 내적은 같은 번호의 성분끼리 곱해 더한 값이고, 행렬 곱의 칸 (i, j)는 i번 행과 j번 열의 내적입니다(`[4, 5, 6] · [2, 1, 0] = 13`). 안쪽 크기가 같아야 하고 결과는 바깥 크기입니다. `nn.Linear`(3차원 @ 2차원 가중치)는 마지막 축만 바꾸고, 3차원끼리의 `@`는 마지막 두 축으로 문장마다 곱합니다.
- **점수 행렬과 softmax:** `Q @ Q.T`는 토큰 쌍마다의 내적 점수 표입니다. softmax는 `e^x`를 합으로 나눠 합 1인 비율로 바꾸고(`[2, 1, 3]` -> `[0.24, 0.09, 0.67]`), 어텐션은 행마다 나누므로 `dim=-1`을 씁니다. 점수를 나누면 비율이 고르게 퍼집니다(2로 나누면 `[0.31, 0.19, 0.51]`).
- **가중합:** 비율 행렬 @ 토큰 벡터 행렬의 한 행은 비율만큼 섞은 벡터입니다(`[0.24, 0.09, 0.67]` -> `[0.91, 0.76, 0.91, 0.09]`). 어텐션 출력이 다시 `(B, W_L, D)`인 이유입니다.
- **transpose:** 두 축만 바꿉니다. 어텐션 점수는 `Q @ K.transpose(-2, -1)`: (B, W_L, D) @ (B, D, W_L) -> (B, W_L, W_L)입니다. W_L = D이면 빠뜨려도 에러가 나지 않으므로 작은 서로 다른 크기로 확인합니다.
- **reshape:** 원소 순서를 지키며 다시 자릅니다. `M.reshape(4, 3)`과 `M.T`는 모양이 같아도 값이 다릅니다.
- **squeeze, unsqueeze:** 크기 1인 축을 빼고 넣습니다. squeeze에는 dim을 적어 배치 축이 사라지지 않게 합니다.